In [1]:
# =============================================================================
# CELL 1 — Install Dependencies
# =============================================================================
# Purpose : Install ML libraries required for XGBoost training and SHAP
#           explainability. MLflow is pre-installed in Fabric notebooks.
#
# Libraries:
#   - xgboost : Gradient boosted trees — binary classification
#   - shap    : SHapley Additive exPlanations — model explainability
#               SHAP values stored in Gold for AI agent consumption
#   - scikit-learn : train_test_split, metrics (accuracy, f1, roc_auc)
# =============================================================================

%pip install xgboost shap scikit-learn --quiet

print("✅ ML libraries installed")

StatementMeta(, 41532cc7-f3f6-4681-863d-d3caa243d1ad, 8, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ ML libraries installed



In [2]:
# =============================================================================
# CELL 2 — Imports & Configuration
# =============================================================================
# Purpose : Import all ML libraries and define training configuration.
#
# Spark Optimizations:
#   - persist(MEMORY_AND_DISK) on feature DataFrame — read twice
#     (once for train/test split, once for SHAP scoring)
#   - toPandas() only at model boundary — all feature prep stays in Spark
#     until final hand-off to XGBoost avoiding driver OOM on larger datasets
#   - cache() on predictions DataFrame — written to Delta + used for SHAP join
#
# MLflow:
#   - Fabric-native MLflow — no external tracking server needed
#   - Tracks: params, metrics, model artifact, feature importance
#   - Experiment name: pulsegrid_spike_predictor
# =============================================================================

import mlflow
import mlflow.xgboost
import xgboost as xgb
import shap
import pandas as pd
import numpy as np

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType, BooleanType, IntegerType
)
from delta.tables import DeltaTable

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    roc_auc_score, classification_report,
    confusion_matrix
)

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
GOLD_FEATURES_PATH    = "Tables/gold_price_features"
GOLD_PREDICTIONS_PATH = "Tables/gold_price_predictions"
GOLD_SHAP_PATH        = "Tables/gold_shap_values"

# -----------------------------------------------------------------------------
# Training Configuration
# -----------------------------------------------------------------------------
TARGET_COL   = "is_spike"
TEST_SIZE    = 0.2
RANDOM_STATE = 42

# Feature columns used for training
# Excludes: identifiers, target, partition cols, and derived labels
FEATURE_COLS = [
    "price_eur_mwh",
    "price_lag_1h",
    "price_lag_12h",
    "price_lag_24h",
    "price_rolling_avg_6h",
    "price_rolling_std_6h",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "temperature_c",
    "wind_speed_ms",
    "humidity_pct",
    "solar_radiation",
    "load_mw",
]

# XGBoost hyperparameters
XGBOOST_PARAMS = {
    "n_estimators"      : 100,
    "max_depth"         : 4,
    "learning_rate"     : 0.1,
    "subsample"         : 0.8,
    "colsample_bytree"  : 0.8,
    "min_child_weight"  : 5,
    "scale_pos_weight"  : 1,   # adjust if class imbalance detected
    "eval_metric"       : "logloss",
    "use_label_encoder" : False,
    "random_state"      : RANDOM_STATE,
}

# MLflow experiment
EXPERIMENT_NAME = "pulsegrid_spike_predictor"
mlflow.set_experiment(EXPERIMENT_NAME)

print("✅ Imports and config loaded")
print(f"   Features         : {len(FEATURE_COLS)}")
print(f"   Target           : {TARGET_COL}")
print(f"   Test size        : {TEST_SIZE}")
print(f"   MLflow experiment: {EXPERIMENT_NAME}")

StatementMeta(, 41532cc7-f3f6-4681-863d-d3caa243d1ad, 10, Finished, Available, Finished, False)

Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
2026/08/13 13:35:11 INFO mlflow.tracking.fluent: Experiment with name 'pulsegrid_spike_predictor' does not exist. Creating a new experiment.


✅ Imports and config loaded
   Features         : 14
   Target           : is_spike
   Test size        : 0.2
   MLflow experiment: pulsegrid_spike_predictor


In [3]:
# =============================================================================
# CELL 3 — Load & Prepare Feature Data
# =============================================================================
# Purpose : Load gold_price_features, handle nulls, and prepare for training.
#
# Spark Optimization — persist(MEMORY_AND_DISK):
#   - Feature DataFrame is read twice:
#     1. toPandas() for model training
#     2. Join with SHAP values for Gold write
#   - persist() caches in memory, spills to disk if Trial cluster memory
#     is insufficient — safe for all cluster sizes
#
# Spark Optimization — toPandas() only at model boundary:
#   - All null handling, casting, filtering done in Spark
#   - Only the final clean DataFrame crosses to pandas for XGBoost
#   - Keeps large intermediate operations distributed
#
# Null handling strategy:
#   - Lag features (price_lag_1h/12h/24h): fill with current price
#     (first records have no history — filling with current is neutral)
#   - Rolling features: fill with current price
#   - Weather/load: fill with 0.0 (sensor outage fallback)
#   - is_weekend, is_holiday: cast Boolean to Integer for XGBoost
# =============================================================================

# Load Gold feature table
df_features = spark.read.format("delta").load(GOLD_FEATURES_PATH)

print(f"✅ Gold features loaded: {df_features.count()} rows")
print(f"   Columns: {len(df_features.columns)}")

# Handle nulls — all in Spark before toPandas()
df_clean = df_features \
    .withColumn("price_lag_1h",
        F.coalesce(F.col("price_lag_1h"), F.col("price_eur_mwh"))
    ) \
    .withColumn("price_lag_12h",
        F.coalesce(F.col("price_lag_12h"), F.col("price_eur_mwh"))
    ) \
    .withColumn("price_lag_24h",
        F.coalesce(F.col("price_lag_24h"), F.col("price_eur_mwh"))
    ) \
    .withColumn("price_rolling_avg_6h",
        F.coalesce(F.col("price_rolling_avg_6h"), F.col("price_eur_mwh"))
    ) \
    .withColumn("price_rolling_std_6h",
        F.coalesce(F.col("price_rolling_std_6h"), F.lit(0.0))
    ) \
    .withColumn("temperature_c",
        F.coalesce(F.col("temperature_c"), F.lit(0.0))
    ) \
    .withColumn("wind_speed_ms",
        F.coalesce(F.col("wind_speed_ms"), F.lit(0.0))
    ) \
    .withColumn("humidity_pct",
        F.coalesce(F.col("humidity_pct"), F.lit(0.0))
    ) \
    .withColumn("solar_radiation",
        F.coalesce(F.col("solar_radiation"), F.lit(0.0))
    ) \
    .withColumn("load_mw",
        F.coalesce(F.col("load_mw"), F.lit(0.0))
    ) \
    .withColumn("is_weekend",
        F.col("is_weekend").cast("integer")
    ) \
    .withColumn("is_holiday",
        F.col("is_holiday").cast("integer")
    ) \
    .withColumn("is_spike",
        F.col("is_spike").cast("integer")
    )

# Persist — read twice (training + SHAP join)
df_clean.persist()

# Select only feature + target + identifier columns
df_ml = df_clean.select(
    ["region", "event_time"] + FEATURE_COLS + [TARGET_COL]
)

# Convert to pandas at model boundary — crossing to XGBoost
pdf = df_ml.toPandas()

print(f"✅ Feature DataFrame prepared")
print(f"   Total rows    : {len(pdf)}")
print(f"   Spike count   : {pdf[TARGET_COL].sum()}")
print(f"   Non-spike     : {(pdf[TARGET_COL] == 0).sum()}")
print(f"   Null check    : {pdf[FEATURE_COLS].isnull().sum().sum()} nulls remaining")

StatementMeta(, 41532cc7-f3f6-4681-863d-d3caa243d1ad, 11, Finished, Available, Finished, False)

✅ Gold features loaded: 30 rows
   Columns: 23
✅ Feature DataFrame prepared
   Total rows    : 30
   Spike count   : 0
   Non-spike     : 30
   Null check    : 0 nulls remaining


In [4]:
# =============================================================================
# CELL 4 — Train/Test Split
# =============================================================================
# Purpose : Split feature DataFrame into train and test sets.
#
# Strategy — time-aware split:
#   - Sort by event_time before splitting
#   - Older records → training, newer records → testing
#   - Prevents data leakage: model never sees future data during training
#   - More realistic evaluation than random split on time-series data
#
# Note on seed data:
#   - With 30 seed records and 0 spikes, metrics will show 100% accuracy
#     trivially (predicting all 0s is correct when no spikes exist)
#   - This is expected — live data will produce real spike distribution
#   - Model architecture and MLflow tracking are validated regardless
# =============================================================================

# Sort by event_time — time-aware split
pdf_sorted = pdf.sort_values("event_time").reset_index(drop=True)

X = pdf_sorted[FEATURE_COLS]
y = pdf_sorted[TARGET_COL]

# Time-aware split — last 20% as test set
split_idx  = int(len(pdf_sorted) * (1 - TEST_SIZE))
X_train    = X.iloc[:split_idx]
X_test     = X.iloc[split_idx:]
y_train    = y.iloc[:split_idx]
y_test     = y.iloc[split_idx:]

# Keep identifiers for prediction write-back
ids_test   = pdf_sorted[["region", "event_time"]].iloc[split_idx:]

print(f"✅ Train/test split complete (time-aware)")
print(f"   Train rows : {len(X_train)}")
print(f"   Test rows  : {len(X_test)}")
print(f"   Train spikes: {y_train.sum()}")
print(f"   Test spikes : {y_test.sum()}")

StatementMeta(, 41532cc7-f3f6-4681-863d-d3caa243d1ad, 12, Finished, Available, Finished, False)

✅ Train/test split complete (time-aware)
   Train rows : 24
   Test rows  : 6
   Train spikes: 0
   Test spikes : 0


In [5]:
# =============================================================================
# CELL 5 — Train XGBoost + MLflow Experiment Tracking
# =============================================================================
# Purpose : Train XGBoost binary classifier and log everything to MLflow.
#
# MLflow tracking (Fabric-native):
#   - Parameters : all XGBoost hyperparameters + feature count + data sizes
#   - Metrics    : accuracy, f1, roc_auc, confusion matrix values
#   - Model      : XGBoost model artifact (loadable for inference)
#   - Tags       : phase, dataset, model_type for experiment organization
#
# XGBoost configuration:
#   - scale_pos_weight: adjusted if class imbalance detected
#     (spike_count / non_spike_count ratio)
#   - eval_metric: logloss — probabilistic, better than error for imbalanced
#   - max_depth=4: shallow trees — prevents overfitting on small dataset
#   - min_child_weight=5: regularization — requires 5 samples per leaf
# =============================================================================

# Adjust scale_pos_weight for class imbalance
spike_count     = int(y_train.sum())
non_spike_count = int((y_train == 0).sum())

if spike_count > 0:
    scale_pos_weight = non_spike_count / spike_count
else:
    scale_pos_weight = 1  # No spikes in seed data — neutral weight

XGBOOST_PARAMS["scale_pos_weight"] = scale_pos_weight

print(f"   Class balance — Spikes: {spike_count}, Non-spike: {non_spike_count}")
print(f"   scale_pos_weight set to: {scale_pos_weight:.2f}")

# -----------------------------------------------------------------------------
# MLflow Run
# -----------------------------------------------------------------------------
with mlflow.start_run(run_name="xgboost_spike_predictor_v1") as run:

    # Log parameters
    mlflow.log_params(XGBOOST_PARAMS)
    mlflow.log_param("feature_count",  len(FEATURE_COLS))
    mlflow.log_param("train_rows",     len(X_train))
    mlflow.log_param("test_rows",      len(X_test))
    mlflow.log_param("split_strategy", "time_aware")
    mlflow.log_param("target_col",     TARGET_COL)

    # Tags for experiment organization
    mlflow.set_tags({
        "phase"       : "phase4_ml",
        "dataset"     : "gold_price_features",
        "model_type"  : "xgboost_binary_classifier",
        "project"     : "pulsegrid"
    })

    # Train XGBoost
    model = xgb.XGBClassifier(**XGBOOST_PARAMS)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    # Predictions
    y_pred      = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1       = f1_score(y_test, y_pred, zero_division=0)

    # roc_auc requires both classes present
    try:
        roc_auc = roc_auc_score(y_test, y_pred_prob)
    except ValueError:
        roc_auc = 0.0
        print("   ⚠️  ROC AUC undefined — only one class in test set (expected with seed data)")

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (cm[0][0], 0, 0, 0)

    # Log metrics
    mlflow.log_metrics({
        "accuracy"  : accuracy,
        "f1_score"  : f1,
        "roc_auc"   : roc_auc,
        "true_neg"  : int(tn),
        "false_pos" : int(fp),
        "false_neg" : int(fn),
        "true_pos"  : int(tp),
    })

    # Log feature importance
    feat_importance = dict(zip(FEATURE_COLS, model.feature_importances_))
    for feat, imp in feat_importance.items():
        mlflow.log_metric(f"importance_{feat}", float(imp))

    # Log model artifact
    mlflow.xgboost.log_model(model, artifact_path="xgboost_model")

    # Store run ID for later reference
    RUN_ID    = run.info.run_id
    MODEL_URI = f"runs:/{RUN_ID}/xgboost_model"

print(f"\n✅ XGBoost training complete")
print(f"   MLflow Run ID : {RUN_ID}")
print(f"   Accuracy      : {accuracy:.4f}")
print(f"   F1 Score      : {f1:.4f}")
print(f"   ROC AUC       : {roc_auc:.4f}")
print(f"\n   Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))
print(f"\n   Top 5 Features by Importance:")
top_features = sorted(feat_importance.items(), key=lambda x: x[1], reverse=True)[:5]
for feat, imp in top_features:
    print(f"   {feat:<30} {imp:.4f}")

StatementMeta(, 41532cc7-f3f6-4681-863d-d3caa243d1ad, 13, Finished, Available, Finished, False)

   Class balance — Spikes: 0, Non-spike: 24
   scale_pos_weight set to: 1.00
   ⚠️  ROC AUC undefined — only one class in test set (expected with seed data)


[13:36:22] WARNING: /croot/xgboost-split_1713972711803/work/cpp_src/src/c_api/c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.
Setuptools is replacing distutils.



✅ XGBoost training complete
   MLflow Run ID : a9024327-d3d0-43f5-96b2-e9ba2ce01ea8
   Accuracy      : 1.0000
   F1 Score      : 0.0000
   ROC AUC       : 0.0000

   Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         6

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6


   Top 5 Features by Importance:
   price_eur_mwh                  0.0000
   price_lag_1h                   0.0000
   price_lag_12h                  0.0000
   price_lag_24h                  0.0000
   price_rolling_avg_6h           0.0000


In [6]:
# =============================================================================
# CELL 6 — SHAP Explainability
# =============================================================================
# Purpose : Compute SHAP values for all test predictions and store in Gold.
#           SHAP values are the primary explainability signal consumed by
#           the Claude AI agent in Phase 6.
#
# SHAP (SHapley Additive exPlanations):
#   - Assigns each feature a contribution value for each prediction
#   - Positive SHAP value → pushed prediction toward spike (1)
#   - Negative SHAP value → pushed prediction toward non-spike (0)
#   - Sum of all SHAP values = model output (log-odds)
#
# Storage strategy:
#   - One row per (region, event_time, feature)
#   - Long format — easier for AI agent to query top contributors
#   - Stored in gold_shap_values Delta table
#
# Spark Optimization — persist() on df_clean:
#   - Already persisted in Cell 3
#   - Joined here with SHAP pandas results → avoids re-reading Gold Delta
# =============================================================================

# Compute SHAP values on test set
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Convert to DataFrame — long format (one row per feature per prediction)
shap_records = []
for i, (_, row) in enumerate(ids_test.iterrows()):
    for j, feat in enumerate(FEATURE_COLS):
        shap_records.append({
            "region"      : row["region"],
            "event_time"  : row["event_time"],
            "feature_name": feat,
            "shap_value"  : float(shap_values[i][j]),
            "feature_value": float(X_test.iloc[i][feat])
        })

# Convert to Spark DataFrame
pdf_shap = pd.DataFrame(shap_records)
df_shap  = spark.createDataFrame(pdf_shap)

# Write to Gold SHAP table
if DeltaTable.isDeltaTable(spark, GOLD_SHAP_PATH):
    DeltaTable.forPath(spark, GOLD_SHAP_PATH).alias("t").merge(
        df_shap.alias("s"),
        "t.region = s.region AND t.event_time = s.event_time AND t.feature_name = s.feature_name"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("✅ gold_shap_values — MERGE complete")
else:
    df_shap.write \
        .format("delta") \
        .mode("overwrite") \
        .save(GOLD_SHAP_PATH)
    print("✅ gold_shap_values — created")

print(f"   SHAP records written : {df_shap.count()}")
print(f"   Features tracked     : {len(FEATURE_COLS)}")
print(f"   Records explained    : {len(ids_test)}")

StatementMeta(, 41532cc7-f3f6-4681-863d-d3caa243d1ad, 14, Finished, Available, Finished, False)

[13:37:05] WARNING: /croot/xgboost-split_1713972711803/work/cpp_src/src/c_api/c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.


✅ gold_shap_values — created
   SHAP records written : 84
   Features tracked     : 14
   Records explained    : 6


In [7]:
# =============================================================================
# CELL 7 — Write Predictions to Gold
# =============================================================================
# Purpose : Write model predictions + probabilities back to a Gold Delta
#           table (gold_price_predictions) for Power BI and AI agent.
#
# Prediction schema:
#   - region, event_time     : identifiers
#   - predicted_spike        : binary prediction (0/1)
#   - spike_probability      : model confidence (0.0 → 1.0)
#   - actual_spike           : ground truth label
#   - prediction_correct     : True if predicted == actual
#   - model_run_id           : MLflow run ID for traceability
# =============================================================================

# Build predictions DataFrame
pdf_preds = ids_test.copy()
pdf_preds["predicted_spike"]    = y_pred.tolist()
pdf_preds["spike_probability"]  = y_pred_prob.tolist()
pdf_preds["actual_spike"]       = y_test.values.tolist()
pdf_preds["prediction_correct"] = (y_pred == y_test.values).tolist()
pdf_preds["model_run_id"]       = RUN_ID

# Convert to Spark DataFrame
df_predictions = spark.createDataFrame(pdf_preds) \
    .withColumn("event_time", F.to_timestamp("event_time")) \
    .withColumn("year",  F.year("event_time")) \
    .withColumn("month", F.month("event_time")) \
    .withColumn("day",   F.dayofmonth("event_time"))

# Write to Gold predictions table
if DeltaTable.isDeltaTable(spark, GOLD_PREDICTIONS_PATH):
    DeltaTable.forPath(spark, GOLD_PREDICTIONS_PATH).alias("t").merge(
        df_predictions.alias("s"),
        "t.region = s.region AND t.event_time = s.event_time"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("✅ gold_price_predictions — MERGE complete")
else:
    df_predictions.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("year", "month", "day") \
        .save(GOLD_PREDICTIONS_PATH)
    print("✅ gold_price_predictions — created")

print(f"   Predictions written  : {df_predictions.count()}")
print(f"   Correct predictions  : {df_predictions.filter(F.col('prediction_correct')==True).count()}")
print(f"   MLflow Run ID        : {RUN_ID}")

# Unpersist cached DataFrame
df_clean.unpersist()
print("✅ Feature cache released")

StatementMeta(, 41532cc7-f3f6-4681-863d-d3caa243d1ad, 15, Finished, Available, Finished, False)

✅ gold_price_predictions — created
   Predictions written  : 6
   Correct predictions  : 6
   MLflow Run ID        : a9024327-d3d0-43f5-96b2-e9ba2ce01ea8
✅ Feature cache released


In [8]:
# =============================================================================
# CELL 8 — ML Layer Validation
# =============================================================================
# Purpose : Validate all ML outputs — predictions, SHAP values, and
#           MLflow experiment. Final gate before Phase 5 Power BI.
# =============================================================================

print("=" * 55)
print("  Phase 4 ML — Validation Report")
print("=" * 55)

# Predictions
df_pred_val = spark.read.format("delta").load(GOLD_PREDICTIONS_PATH)
print(f"\n  ✅ gold_price_predictions")
print(f"     Total rows           : {df_pred_val.count()}")
print(f"     Predicted spikes     : {df_pred_val.filter(F.col('predicted_spike')==1).count()}")
print(f"     Correct predictions  : {df_pred_val.filter(F.col('prediction_correct')==True).count()}")
df_pred_val.select(
    "region", "event_time",
    "predicted_spike", "spike_probability",
    "actual_spike", "prediction_correct"
).show(5, truncate=False)

# SHAP values
df_shap_val = spark.read.format("delta").load(GOLD_SHAP_PATH)
print(f"\n  ✅ gold_shap_values")
print(f"     Total SHAP records   : {df_shap_val.count()}")
print(f"     Features tracked     : {df_shap_val.select('feature_name').distinct().count()}")

# Top SHAP contributors
print(f"\n     Top features by mean |SHAP|:")
df_shap_val \
    .withColumn("abs_shap", F.abs(F.col("shap_value"))) \
    .groupBy("feature_name") \
    .agg(F.avg("abs_shap").alias("mean_abs_shap")) \
    .orderBy(F.col("mean_abs_shap").desc()) \
    .show(5, truncate=False)

# MLflow
print(f"\n  ✅ MLflow Experiment")
print(f"     Experiment : {EXPERIMENT_NAME}")
print(f"     Run ID     : {RUN_ID}")
print(f"     Model URI  : {MODEL_URI}")
print(f"     Accuracy   : {accuracy:.4f}")
print(f"     F1 Score   : {f1:.4f}")
print(f"     ROC AUC    : {roc_auc:.4f}")

print(f"\n{'='*55}")
print(f"  Phase 4 ML — Complete ✅")
print(f"{'='*55}")

StatementMeta(, 41532cc7-f3f6-4681-863d-d3caa243d1ad, 16, Finished, Available, Finished, False)

  Phase 4 ML — Validation Report

  ✅ gold_price_predictions
     Total rows           : 6
     Predicted spikes     : 0
     Correct predictions  : 6
+------+-------------------+---------------+-------------------+------------+------------------+
|region|event_time         |predicted_spike|spike_probability  |actual_spike|prediction_correct|
+------+-------------------+---------------+-------------------+------------+------------------+
|ES    |2026-08-13 10:00:00|0              |0.11920291930437088|0           |true              |
|FR    |2026-08-13 10:00:00|0              |0.11920291930437088|0           |true              |
|DE    |2026-08-13 10:00:00|0              |0.11920291930437088|0           |true              |
|BE    |2026-08-13 10:00:00|0              |0.11920291930437088|0           |true              |
|NL    |2026-08-13 10:00:00|0              |0.11920291930437088|0           |true              |
+------+-------------------+---------------+-------------------+---------